In [1]:
# --- DEBUG: FLA model dims (mirrors run_fla_on_kv_retrieval-default.py) ---
from pathlib import Path
import importlib.util
from types import SimpleNamespace

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:

# >>> tweak these
ROOT = Path("/home/bulatov/rmt/test-time/compressing-associations").resolve()
TOKENIZER_PATH = str(ROOT / "tokenizers/kv_alphabet_62")

BASE_MODEL = "gated_delta_net"   # or "mamba", "mamba2"
N_LAYER = 4
N_HEAD = 1
N_EMBD = 128
STATE_SIZE = 16
CONV_KERNEL = 4

# Load script module (filename has a hyphen → cannot `import run_fla_on_kv_retrieval-default`)
_spec = importlib.util.spec_from_file_location(
    "_run_fla_kv_default",
    ROOT / "run_fla_on_kv_retrieval-default.py",
)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
build_fla_config = _mod.build_fla_config

2026-05-02 12:42:52,018 - root - INFO - CUDA DEVICE COUNT: 2


In [3]:

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

args = SimpleNamespace(
    n_layer=N_LAYER,
    n_head=N_HEAD,
    n_embd=N_EMBD,
    state_size=STATE_SIZE,
    conv_kernel=CONV_KERNEL,
)

cfg = build_fla_config(BASE_MODEL, args, tokenizer)
model = AutoModelForCausalLM.from_config(cfg)
model.eval()

print("=== Saved config (subset) ===")
for k in sorted(
    "vocab_size hidden_size num_hidden_layers num_heads head_dim "
    "state_size conv_kernel conv_size expand_v attn_mode use_gate use_short_conv "
    "pad_token_id".split()
):
    if hasattr(cfg, k):
        print(f"  {k}: {getattr(cfg, k)}")

print("\n=== Derived (training-script convention for GDN) ===")
if BASE_MODEL == "gated_delta_net":
    hd = getattr(cfg, "head_dim", None)
    nh = getattr(cfg, "num_heads", None)
    ev = getattr(cfg, "expand_v", None)
    print(f"  CLI state_size used only to set head_dim = state_size // n_head → {STATE_SIZE}//{N_HEAD} = {hd}")
    if hd is not None and nh is not None and ev is not None:
        print(f"  key_dim (num_heads * head_dim) = {nh} * {hd} = {nh * hd}")
        print(f"  head_v_dim ≈ head_dim * expand_v = {hd} * {ev} = {int(hd * ev)}")
        print(f"  value_dim (num_v_heads * head_v_dim) = {nh} * {int(hd * ev)} = {nh * int(hd * ev)}")

print("\n=== First layer attention submodule ===")
blk = model.model.layers[0]
attn = blk.attn
print(f"  type: {type(attn).__name__}")

if type(attn).__name__ == "GatedDeltaNet":
    print(f"  hidden_size={attn.hidden_size}, num_heads={attn.num_heads}, head_dim={attn.head_dim}")
    print(f"  head_k_dim={attn.head_k_dim}, head_v_dim={attn.head_v_dim}")
    print(f"  key_dim={attn.key_dim}, value_dim={attn.value_dim}")
    print("  Linear projections:")
    for name in ("q_proj", "k_proj", "v_proj", "o_proj", "g_proj"):
        if hasattr(attn, name):
            L = getattr(attn, name)
            print(f"    {name}: ({L.in_features} → {L.out_features})")

=== Saved config (subset) ===
  attn_mode: chunk
  conv_size: 4
  expand_v: 2.0
  head_dim: 16
  hidden_size: 128
  num_heads: 1
  num_hidden_layers: 4
  pad_token_id: 0
  use_gate: True
  use_short_conv: True
  vocab_size: 70

=== Derived (training-script convention for GDN) ===
  CLI state_size used only to set head_dim = state_size // n_head → 16//1 = 16
  key_dim (num_heads * head_dim) = 1 * 16 = 16
  head_v_dim ≈ head_dim * expand_v = 16 * 2.0 = 32
  value_dim (num_v_heads * head_v_dim) = 1 * 32 = 32

=== First layer attention submodule ===
  type: GatedDeltaNet
  hidden_size=128, num_heads=1, head_dim=16
  head_k_dim=16, head_v_dim=32
  key_dim=16, value_dim=32
  Linear projections:
    q_proj: (128 → 16)
    k_proj: (128 → 16)
    v_proj: (128 → 32)
    o_proj: (32 → 128)
    g_proj: (128 → 32)


In [4]:
# --- DEBUG: FLA model dims (mirrors run_fla_on_kv_retrieval-default.py) ---
from pathlib import Path
import importlib.util
from types import SimpleNamespace

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# >>> tweak these
ROOT = Path("/home/bulatov/rmt/test-time/compressing-associations").resolve()
TOKENIZER_PATH = str(ROOT / "tokenizers/kv_alphabet_62")

BASE_MODEL = "gated_delta_net"   # or "mamba", "mamba2"
N_LAYER = 4
N_HEAD = 1
N_EMBD = 128
STATE_SIZE = 16
CONV_KERNEL = 4

# Load script module (filename has a hyphen → cannot `import run_fla_on_kv_retrieval-default`)
_spec = importlib.util.spec_from_file_location(
    "_run_fla_kv_default",
    ROOT / "run_fla_on_kv_retrieval-default.py",
)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
build_fla_config = _mod.build_fla_config

2026-05-02 12:42:54,926 - root - INFO - CUDA DEVICE COUNT: 2


In [5]:

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

args = SimpleNamespace(
    n_layer=N_LAYER,
    n_head=N_HEAD,
    n_embd=N_EMBD,
    state_size=STATE_SIZE,
    conv_kernel=CONV_KERNEL,
)

cfg = build_fla_config(BASE_MODEL, args, tokenizer)
model = AutoModelForCausalLM.from_config(cfg)
model.eval()

print("=== Saved config (subset) ===")
for k in sorted(
    "vocab_size hidden_size num_hidden_layers num_heads head_dim "
    "state_size conv_kernel conv_size expand_v attn_mode use_gate use_short_conv "
    "pad_token_id".split()
):
    if hasattr(cfg, k):
        print(f"  {k}: {getattr(cfg, k)}")

print("\n=== Derived (training-script convention for GDN) ===")
if BASE_MODEL == "gated_delta_net":
    hd = getattr(cfg, "head_dim", None)
    nh = getattr(cfg, "num_heads", None)
    ev = getattr(cfg, "expand_v", None)
    print(f"  CLI state_size used only to set head_dim = state_size // n_head → {STATE_SIZE}//{N_HEAD} = {hd}")
    if hd is not None and nh is not None and ev is not None:
        print(f"  key_dim (num_heads * head_dim) = {nh} * {hd} = {nh * hd}")
        print(f"  head_v_dim ≈ head_dim * expand_v = {hd} * {ev} = {int(hd * ev)}")
        print(f"  value_dim (num_v_heads * head_v_dim) = {nh} * {int(hd * ev)} = {nh * int(hd * ev)}")

print("\n=== First layer attention submodule ===")
blk = model.model.layers[0]
attn = blk.attn
print(f"  type: {type(attn).__name__}")

if type(attn).__name__ == "GatedDeltaNet":
    print(f"  hidden_size={attn.hidden_size}, num_heads={attn.num_heads}, head_dim={attn.head_dim}")
    print(f"  head_k_dim={attn.head_k_dim}, head_v_dim={attn.head_v_dim}")
    print(f"  key_dim={attn.key_dim}, value_dim={attn.value_dim}")
    print("  Linear projections:")
    for name in ("q_proj", "k_proj", "v_proj", "o_proj", "g_proj"):
        if hasattr(attn, name):
            L = getattr(attn, name)
            print(f"    {name}: ({L.in_features} → {L.out_features})")

# # Optional: one forward step to ensure shapes match
# with torch.no_grad():
#     x = torch.randint(0, tokenizer.vocab_size, (2, 32))
#     out = model(x)
# print("\n=== Smoke forward ===")
# print(f"  logits shape: {tuple(out.logits.shape)}")

=== Saved config (subset) ===
  attn_mode: chunk
  conv_size: 4
  expand_v: 2.0
  head_dim: 16
  hidden_size: 128
  num_heads: 1
  num_hidden_layers: 4
  pad_token_id: 0
  use_gate: True
  use_short_conv: True
  vocab_size: 70

=== Derived (training-script convention for GDN) ===
  CLI state_size used only to set head_dim = state_size // n_head → 16//1 = 16
  key_dim (num_heads * head_dim) = 1 * 16 = 16
  head_v_dim ≈ head_dim * expand_v = 16 * 2.0 = 32
  value_dim (num_v_heads * head_v_dim) = 1 * 32 = 32

=== First layer attention submodule ===
  type: GatedDeltaNet
  hidden_size=128, num_heads=1, head_dim=16
  head_k_dim=16, head_v_dim=32
  key_dim=16, value_dim=32
  Linear projections:
    q_proj: (128 → 16)
    k_proj: (128 → 16)
    v_proj: (128 → 32)
    o_proj: (32 → 128)
    g_proj: (128 → 32)


In [7]:
# Only three knobs: state_size, num_heads, head_dim — saved (config) vs actual (mixer).
from pathlib import Path
import importlib.util
from types import SimpleNamespace

from transformers import AutoTokenizer, AutoModelForCausalLM

ROOT = Path("/home/bulatov/rmt/test-time/compressing-associations").resolve()
TOKENIZER_PATH = str(ROOT / "tokenizers/kv_alphabet_62")

N_LAYER, N_EMBD = 4, 128
STATE_SIZE, N_HEAD, CONV_KERNEL = 16, 1, 4

_spec = importlib.util.spec_from_file_location("_rf", ROOT / "run_fla_on_kv_retrieval-default.py")
_m = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_m)
build_fla_config = _m.build_fla_config

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

args = SimpleNamespace(
    n_layer=N_LAYER,
    n_head=N_HEAD,
    n_embd=N_EMBD,
    state_size=STATE_SIZE,
    conv_kernel=CONV_KERNEL,
)


def mixer0(model):
    inner = model.backbone if hasattr(model, "backbone") else model.model
    blk = inner.layers[0]
    return blk.mixer if hasattr(blk, "mixer") else blk.attn


def triple(cfg, mixer, bm):
    """Returns (state_saved, state_act), (nh_saved, nh_act), (hd_saved, hd_act)."""
    if bm == "mamba":
        ss = (getattr(cfg, "state_size", None), getattr(mixer, "ssm_state_size", None))
        return ss, ("—", "—"), ("—", "—")
    if bm == "mamba2":
        ss = (getattr(cfg, "state_size", None), getattr(mixer, "ssm_state_size", None))
        nh = (getattr(cfg, "num_heads", None), getattr(mixer, "num_heads", None))
        hd = (getattr(cfg, "head_dim", None), getattr(mixer, "head_dim", None))
        return ss, nh, hd
    # gated_delta_net — no state_size on config; CLI state_size only affects head_dim via build_fla_config
    ss = ("—", "—")
    nh = (getattr(cfg, "num_heads", None), getattr(mixer, "num_heads", None))
    hd = (getattr(cfg, "head_dim", None), getattr(mixer, "head_dim", None))
    return ss, nh, hd


models_order = ("mamba", "mamba2", "gated_delta_net")
col_w = 8
line_w = 18 + 3 + (2 * col_w + 1) * 3 + 6

print(f"CLI: --state_size {STATE_SIZE} --n_head {N_HEAD} --n_embd {N_EMBD} --conv_kernel {CONV_KERNEL}")
print(
    "Note: Mamba has no heads. Mamba2 + GDN use --n_head; Mamba2 sets head_dim = (expand*n_embd)//n_head.\n"
)

hdr = (
    f"{'model':<18} │ "
    f"{'state_size':^{2 * col_w + 1}} │ "
    f"{'num_heads':^{2 * col_w + 1}} │ "
    f"{'head_dim':^{2 * col_w + 1}}"
)
sub = (
    f"{'':18} │ "
    f"{'saved':<{col_w}} {'act':<{col_w}} │ "
    f"{'saved':<{col_w}} {'act':<{col_w}} │ "
    f"{'saved':<{col_w}} {'act':<{col_w}}"
)
print(hdr)
print(sub)
print("─" * line_w)

for bm in models_order:
    cfg = build_fla_config(bm, args, tokenizer)
    model = AutoModelForCausalLM.from_config(cfg)
    mx = mixer0(model)
    (ss_s, ss_a), (nh_s, nh_a), (hd_s, hd_a) = triple(cfg, mx, bm)
    print(
        f"{bm:<18} │ "
        f"{str(ss_s):<{col_w}} {str(ss_a):<{col_w}} │ "
        f"{str(nh_s):<{col_w}} {str(nh_a):<{col_w}} │ "
        f"{str(hd_s):<{col_w}} {str(hd_a):<{col_w}}"
    )

print(
    "\nGDN: config has no state_size; runner sets head_dim = state_size // n_head → "
    f"{STATE_SIZE}//{N_HEAD} = {STATE_SIZE // N_HEAD}."
)
print("Mamba: no head split — num_heads / head_dim do not apply (—).")

2026-05-02 12:44:11,087 - root - INFO - CUDA DEVICE COUNT: 2


CLI: --state_size 16 --n_head 1 --n_embd 128 --conv_kernel 4
Note: Mamba has no heads. Mamba2 + GDN use --n_head; Mamba2 sets head_dim = (expand*n_embd)//n_head.

model              │    state_size     │     num_heads     │     head_dim     
                   │ saved    act      │ saved    act      │ saved    act     
──────────────────────────────────────────────────────────────────────────────
mamba              │ 16       16       │ —        —        │ —        —       
mamba2             │ 16       16       │ 1        1        │ 256      256     
gated_delta_net    │ —        —        │ 1        1        │ 16       16      

GDN: config has no state_size; runner sets head_dim = state_size // n_head → 16//1 = 16.
Mamba: no head split — num_heads / head_dim do not apply (—).
